In [2]:
import spotipy
import pandas as pd
import math
from spotipy.oauth2 import SpotifyClientCredentials

pd.options.display.max_rows = 999

In [3]:
sp = spotipy.Spotify(
    client_credentials_manager=SpotifyClientCredentials(
        client_id='c3c6cbaa304c4c86b120e8d9603d44a4', 
        client_secret='b8819ea70b3142819602499ee139c24f'))

nirvana_uri = 'spotify:artist:6olE6TJLqED3rqDCT0FyPh'

#### Collecting data from playlist Nirvana all songs (Original)

In [4]:
playlist_id = "spotify:playlist:5PCYPdeq31cVX5pOH2vkpV"

offset = 0
limit = 100

all_tracks = []

while(True):
    results = sp.playlist_tracks(playlist_id, offset=offset, limit=limit)
    
    tracks = results['items']
    
    for track in tracks:
        all_tracks.append(track['track'])
    
    if len(tracks) < limit:
        break
    
    offset += limit


print("Total tracks:", len(all_tracks))


Total tracks: 401


Excluding Home Recordings

In [5]:
all_tracks = all_tracks[43:]
print("Total tracks:", len(all_tracks))
playlist_1 = all_tracks

Total tracks: 358


#### Collecting data from playlist Nirvana Best Of

In [6]:
playlist_id = "spotify:playlist:7ETnL4fbHNcIdV1JmRuCZd"

offset = 0
limit = 100

all_tracks = []

while(True):
    results = sp.playlist_tracks(playlist_id, offset=offset, limit=limit)
    
    tracks = results['items']
    for track in tracks:
        all_tracks.append(track['track'])
    
    if len(tracks) < limit:
        break
    
    offset += limit

print("Total tracks:", len(all_tracks))

Total tracks: 33


In [7]:
playlist_2 = all_tracks

#### Collecting data from playlist This Is Nirvana

In [8]:
playlist_id = "spotify:playlist:37i9dQZF1DZ06evO3M0Fbi"

offset = 0
limit = 100

all_tracks = []

while(True):
    results = sp.playlist_tracks(playlist_id, offset=offset, limit=limit)
    
    tracks = results['items']
    
    for track in tracks:
        all_tracks.append(track['track'])
    
    if len(tracks) < limit:
        break
    
    offset += limit


print("Total tracks:", len(all_tracks))

Total tracks: 39


In [9]:
playlist_3 = all_tracks

### Merging there three playlists

In [10]:
final_list = playlist_1

for track in playlist_2:
    if track not in final_list:
        final_list.append(track)
for track in playlist_3:
    if track not in final_list:
        final_list.append(track)

len(final_list)

392

### Transforming data into a dataset

In [11]:
tracks_dict = {}

track_names = []
album_names = []
album_types = []
duration = []
year = []
artists = []

acousticness = []
danceability = []
energy = []
instrumentalness = []
key = []
liveness = []
loudness = []
mode = []
speechiness = []
tempo = []
time_signature = []
valence = []

for track in final_list:
    audio = sp.audio_features(tracks=track['id'])
    
    track_names.append(track['name'])
    album_names.append(track['album']['name'])
    album_types.append(track['album']['album_type'])
    duration.append(math.floor(track['duration_ms']/1000))
    year.append(track['album']['release_date'][0:4])
    artists.append(''.join([artist['name'] for artist in track['artists']]))
        
    acousticness.append(audio[0]['acousticness'])
    danceability.append(audio[0]['danceability'])
    energy.append(audio[0]['energy'])
    instrumentalness.append(audio[0]['instrumentalness'])
    key.append(audio[0]['key'])
    liveness.append(audio[0]['liveness'])
    loudness.append(audio[0]['loudness'])
    mode.append(audio[0]['mode'])
    speechiness.append(audio[0]['speechiness'])
    tempo.append(audio[0]['tempo'])
    time_signature.append(audio[0]['time_signature'])
    valence.append(audio[0]['valence'])    
        
tracks_dict['track_name'] = track_names
tracks_dict['album_name'] = album_names
tracks_dict['album_type'] = album_types
tracks_dict['duration'] = duration
tracks_dict['release_year'] = year
tracks_dict['artists'] = artists

tracks_dict['acousticness'] = acousticness
tracks_dict['danceability'] = danceability
tracks_dict['energy'] = energy
tracks_dict['instrumentalness'] = instrumentalness
tracks_dict['key'] = key
tracks_dict['liveness'] = liveness
tracks_dict['loudness'] = loudness
tracks_dict['mode'] = mode
tracks_dict['speechiness'] = speechiness
tracks_dict['tempo'] = tempo
tracks_dict['time_signature'] = time_signature
tracks_dict['valence'] = valence

In [12]:
df = pd.DataFrame(tracks_dict)
df.sample(5)

,track_name,album_name,album_type,duration,release_year,artists,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
308,I Hate Myself And Want To Die - Demo,With The Lights Out - Box Set,compilation,242,2004,Nirvana,0.071200,0.256,0.924,0.00971,1,0.0929,-3.755,0,0.0689,115.179,4,0.195
73,Stay Away - Devonshire Mix,Nevermind (Super Deluxe Edition),album,207,1991,Nirvana,0.000123,0.402,0.882,0.59500,0,0.0913,-5.626,1,0.0957,164.398,4,0.442
299,D-7 - 1990 Radio Appearance,With The Lights Out - Box Set,compilation,225,2004,Nirvana,0.034900,0.297,0.923,0.73900,5,0.0891,-4.939,1,0.0779,86.364,1,0.132
20,Dive (Live at Pine Street Theatre),Bleach (Deluxe Edition),album,222,1989,Nirvana,0.000005,0.411,0.895,0.00509,11,0.1400,-5.807,1,0.0342,132.553,4,0.439
15,Molly's Lips (Live at Pine Street Theatre),Bleach (Deluxe Edition),album,135,1989,Nirvana,0.000056,0.284,0.964,0.08030,7,0.2690,-6.651,1,0.1000,156.573,4,0.415


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   track_name        392 non-null    object 
 1   album_name        392 non-null    object 
 2   album_type        392 non-null    object 
 3   duration          392 non-null    int64  
 4   release_year      392 non-null    object 
 5   artists           392 non-null    object 
 6   acousticness      392 non-null    float64
 7   danceability      392 non-null    float64
 8   energy            392 non-null    float64
 9   instrumentalness  392 non-null    float64
 10  key               392 non-null    int64  
 11  liveness          392 non-null    float64
 12  loudness          392 non-null    float64
 13  mode              392 non-null    int64  
 14  speechiness       392 non-null    float64
 15  tempo             392 non-null    float64
 16  time_signature    392 non-null    int64  
 1

In [14]:
df.to_csv('nirvana_v2.csv', index=false)

NameError: name 'false' is not defined